# 5단계: 추천 파이프라인 실험

4단계에서 저장한 e5-base 음식 임베딩(텍스트 A/B)을 그대로 재사용해 자연어 입력에 대한 Top-K 추천을 실험한다.

```
사용자 입력
-> 명시적 선호·제외 조건 추출      src/preprocessing/user_query.py
-> 임베딩 기반 후보 검색            src/retrieval/candidates.py
-> 필수 조건 필터                   src/ranking/rerank.py
-> 선호 일치도 재랭킹               src/ranking/rerank.py
-> 메뉴 중복·다양성 제어            src/ranking/rerank.py
-> Top-K 추천                       src/recommendation/pipeline.py
```

- 입력: `data/embeddings/<e5-base 결과>/` (A, B), `data/processed/labeling/*`, `data/processed/food_menu.csv`
- 출력: `data/processed/recommendation/` (추천 결과, 지표, 파싱 결과, 실행 설정)
- 검증된 로직은 모두 `src/`에 있고 `tests/`로 검증한다. 이 노트북은 실제 데이터에 대해 실행하고 관찰하는 용도다.

주의할 점
- 음식 라벨은 전량 모델 추정(승인 샘플 100개 제외)이다. 아래 조건 준수 지표는 **저장된 추정 라벨 기준**이며 실제 정확도가 아니다.
- 정답 데이터가 없으므로 Precision 같은 정확도 지표는 계산하지 않는다. 점수는 추천 확률이 아니라 정렬용 값이다.
- 재료 데이터가 없어 알레르기·채식 적합성은 다루지 않는다.
- 4단계의 384차원 결과(`embeddings_v1_text*.npy`)는 manifest가 없어 사용하지 않는다.

## 1. 모듈 로드 및 실행 환경

In [1]:
import json
import sys
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding import DEFAULT_SPEC, E5Embedder, EmbeddingStore, describe_environment, source_hashes
from src.preprocessing import parse_query, support_table
from src.ranking import RankingConfig
from src.recommendation import (
    EMBEDDING_ONLY, FILTER_ONLY, FULL, PipelineConfig, Recommender, result_metrics, result_rows,
)
from src.retrieval import check_compatibility, load_index

OUT_DIR = PROJECT_ROOT / "data" / "processed" / "recommendation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

env = describe_environment()
env

{'platform': 'macOS-26.6.2-arm64-arm-64bit',
 'processor': 'arm',
 'cpu_count': 12,
 'total_memory_gb': 32.0,
 'torch_version': '2.14.0',
 'cuda_available': False,
 'mps_available': True,
 'selected_device': 'mps'}

## 2. 저장된 임베딩 확인

manifest의 모델·리비전·차원·접두어와 원본 파일 해시(`labeling_units.csv`, `labels_chat_full_v3.jsonl`, `food_menu.csv`)를
현재 파일과 대조해 호환되는 결과만 로드한다. 불일치가 있으면 여기서 멈춘다. 임베딩을 새로 만들지 않는다.

In [2]:
store = EmbeddingStore()
pd.DataFrame(store.list_results())

,name,model_id,model_revision,text_variant,num_vectors,dimension,created_at
0,embeddings_v1_textA_n5219.npy,NaN,NaN,NaN,5219,384,NaN
1,embeddings_v1_textB_n5219.npy,NaN,NaN,NaN,5219,384,NaN
2,multilingual-e5-base_d1287505_textA_v1_eb332ded,intfloat/multilingual-e5-base,d1287505,A,5219,768,2026-09-22T08:02:55+00:00
3,multilingual-e5-base_d1287505_textB_v1_f87e895f,intfloat/multilingual-e5-base,d1287505,B,5219,768,2026-09-22T08:03:45+00:00


In [3]:
sources = source_hashes()
index_a, ref_a = load_index("A", store, sources=sources)
index_b, ref_b = load_index("B", store, sources=sources)

for ref in (ref_a, ref_b):
    problems = check_compatibility(store.read_manifest(ref["name"]), sources=sources)
    print(f"텍스트 {ref['text_variant']}: {ref['name']}")
    print(f"  {ref['num_vectors']} x {ref['dimension']}, config_hash={ref['config_hash'][:12]}, "
          f"생성={ref['created_at']}, 호환 문제={problems or '없음'}")
assert index_a.size == index_b.size == 5219

텍스트 A: multilingual-e5-base_d1287505_textA_v1_eb332ded
  5219 x 768, config_hash=eb332dedac1f, 생성=2026-09-22T08:02:55+00:00, 호환 문제=없음
텍스트 B: multilingual-e5-base_d1287505_textB_v1_f87e895f
  5219 x 768, config_hash=f87e895f6c65, 생성=2026-09-22T08:03:45+00:00, 호환 문제=없음


## 3. 사용자 조건 파서 지원 범위

규칙 기반 파서이며 아래 표가 지원 범위 전체다. 라벨을 새로 만드는 작업(3단계)과 달리 사용자 문장에서
저장된 라벨 스키마의 속성·값에 대응하는 조건만 뽑는다.

- **필수(hard)**: 부정·제외 표현과 "꼭/반드시/무조건"이 붙은 표현. 허용값 집합이며 '미확인'은 허용값이 아니다.
- **선호(soft)**: 긍정 표현. 점수에만 반영한다.
- **unhandled**: 이중 부정, 허용 표현, 모호한 표현, 스키마에 없는 맛은 조건으로 확정하지 않고 기록만 한다.
- **context**: 날씨·기분·시간대는 조건으로 쓰지 않는다 (원문은 임베딩에 그대로 들어간다).
- "가벼운"은 든든함 라벨 '가벼움'에만 대응하고 칼로리로 해석하지 않는다.

In [4]:
pd.DataFrame(support_table())

,종류,규칙,예시,조건,비고
0,unhandled,이중부정,안 매운 건 싫어,-,이중 부정은 방향을 확정하지 않음
1,unhandled,허용표현,매운 것도 괜찮아,-,허용·관용 표현은 필수·선호 조건으로 확정하지 않음
2,unhandled,시원한국물,시원한 국물,-,'시원한 국물'은 온도가 아닐 수 있어 확정하지 않음
3,unhandled,스키마외맛,단짠단짠한 음식,-,"라벨 스키마에 없는 맛·식감, 임베딩 유사도에만 맡김"
4,hard,매운맛_강함제외,너무 맵지 않은,매운맛∈없음/약함/보통,-
5,hard,매운맛_제외,"맵지 않은, 안 매운, 매운 거 싫어",매운맛∈없음,-
6,hard,국물_제외,"국물 없는, 국물 빼고",국물∈국물없음,-
7,hard,뜨거움_제외,뜨겁지 않은,제공온도∈따뜻함/상온/차가움,-
8,hard,차가움_제외,차갑지 않은,제공온도∈뜨거움/따뜻함/상온,-
9,hard,기름짐_제외,"느끼하지 않은, 기름기 적은",기름짐∈낮음/보통,-


In [5]:
BASE_QUERIES = [
    "비 오는 날 얼큰한 국물 먹고 싶어",
    "맵지 않고 따뜻한 음식",
    "차갑고 가볍게 먹을 메뉴",
    "바삭하고 기름진 음식",
    "든든한 밥 한 끼",
    "국물 없는 매운 음식",
    "상큼하고 시원한 음식",
    "포만감 있는 저녁밥",
    "빠르게 먹을 수 있는 간식",
    "따뜻한 국이나 찌개",
    "느끼하지 않은 담백한 음식",
    "단짠단짠한 음식",
]
EXTRA_QUERIES = {
    "부정": ["매운 거 싫어", "튀김 말고 구운 치킨"],
    "복합": ["피자 먹고 싶은데 느끼하지 않은 걸로", "차가운 국물 요리", "너무 맵지 않은 국물 요리"],
    "모순": ["맵지 않은 매운 음식", "국물 없는 국물 요리"],
    "미확정": ["안 매운 건 싫어", "매운 것도 괜찮아"],
    "빈 입력": [""],
}
ALL_QUERIES = BASE_QUERIES + [q for qs in EXTRA_QUERIES.values() for q in qs]
QUERY_KIND = {q: "기존" for q in BASE_QUERIES}
QUERY_KIND.update({q: kind for kind, qs in EXTRA_QUERIES.items() for q in qs})


def parse_row(q):
    p = parse_query(q)
    fmt = lambda conds: "; ".join(f"{c.attribute}∈{'/'.join(c.allowed)} ← {c.evidence}" for c in conds) or "-"
    return {
        "유형": QUERY_KIND[q], "질의": q or "(빈 입력)", "필수": fmt(p.hard), "선호": fmt(p.soft),
        "메뉴언급": ", ".join(p.menu_terms) or "-",
        "미처리": "; ".join(f"{u['expression']} ({u['reason']})" for u in p.unhandled) or "-",
        "무시": ", ".join(i["expression"] for i in p.ignored) or "-",
        "모순": "; ".join(f"{c['attribute']}: {c['evidence']}" for c in p.contradictions) or "-",
    }


parsed_df = pd.DataFrame([parse_row(q) for q in ALL_QUERIES])
parsed_df

,유형,질의,필수,선호,메뉴언급,미처리,무시,모순
0,기존,비 오는 날 얼큰한 국물 먹고 싶어,-,매운맛∈보통/강함 ← 얼큰한; 제공온도∈뜨거움/따뜻함 ← 얼큰한; 국물∈국물요리 ← 국물,-,-,비 오,-
1,기존,맵지 않고 따뜻한 음식,매운맛∈없음 ← 맵지 않고,제공온도∈뜨거움/따뜻함 ← 따뜻한,-,-,-,-
2,기존,차갑고 가볍게 먹을 메뉴,-,제공온도∈차가움 ← 차갑고; 든든함∈가벼움 ← 가볍게,-,-,-,-
3,기존,바삭하고 기름진 음식,-,기름짐∈높음 ← 기름진; 조리법∈튀김 ← 바삭하고,-,-,-,-
4,기존,든든한 밥 한 끼,-,든든함∈든든함 ← 든든한,밥,-,-,-
5,기존,국물 없는 매운 음식,국물∈국물없음 ← 국물 없는,매운맛∈보통/강함 ← 매운,-,-,-,-
6,기존,상큼하고 시원한 음식,-,제공온도∈차가움 ← 시원한,-,"상큼 (라벨 스키마에 없는 맛·식감, 임베딩 유사도에만 맡김)",-,-
7,기존,포만감 있는 저녁밥,-,든든함∈든든함 ← 포만감,밥,-,저녁,-
8,기존,빠르게 먹을 수 있는 간식,-,든든함∈가벼움 ← 간식,-,-,-,-
9,기존,따뜻한 국이나 찌개,-,"국물∈국물요리 ← 국이, 찌개; 제공온도∈뜨거움/따뜻함 ← 따뜻한",찌개,-,-,-


## 4. 질의 임베딩 모델 로드

문서 임베딩과 같은 모델·리비전으로 사용자 문장만 `query: ` 접두어를 붙여 임베딩한다. 음식 임베딩은 다시 만들지 않는다.

In [6]:
t0 = time.time()
embedder = E5Embedder(spec=DEFAULT_SPEC)
print(f"모델 로드 {time.time() - t0:.1f}초, device={embedder.device}, batch={embedder.batch_size}, "
      f"revision={DEFAULT_SPEC.revision[:8]}")
assert ref_b["model_revision"] == DEFAULT_SPEC.revision

encode_query = lambda text: embedder.encode_queries([text], show_progress=False)[0]
rec_b = Recommender(index_b, encode_query, ref_b)
rec_a = Recommender(index_a, encode_query, ref_a)
_ = rec_b.recommend("워밍업")  # 첫 MPS 호출 지연을 측정에서 제외

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

모델 로드 7.5초, device=mps, batch=32, revision=d1287505


/Users/jack/project/Menu-recommend-algorithmn/src/embedding/embedder.py:123: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_dim = self.model.get_sentence_embedding_dimension()


## 5. 비교 모드 설정

세 모드는 같은 파이프라인의 설정 차이다. 최종점수 = `similarity_weight × 유사도 + preference_weight × 선호점수`.
유사도는 원값(0.8~0.9 대역)이라 선호 가중치 0.3은 "후보 안에서 선호 일치가 유사도 차이보다 우선"이라는 뜻이다. 10절에서 가중치를 바꿔 본다.

- 임베딩만: 필터 없음, 유사도 정렬, 중복 제어 없음 (4단계 검색과 같음)
- 조건적용: 필수 조건 필터만 추가
- 조건+재랭킹+중복제어: 필터 + 선호 재랭킹 + 중복 제거 + 메뉴군 상한 2 (사용자가 언급한 메뉴는 상한 면제). 선호를 모두 만족하는 항목이 Top-K개 미만이면 후보를 400까지 넓힌다.

메뉴군: 프랜차이즈 항목은 대표식품명(피자, 버거) 그대로, 업체명 없는 공공 데이터는 대표식품명 마지막 어절("붕어 매운탕" → "매운탕")로 묶는다.

In [7]:
MODES = {"임베딩만": EMBEDDING_ONLY, "조건적용": FILTER_ONLY, "조건+재랭킹+중복제어": FULL}
pd.DataFrame({
    name: {**{k: v for k, v in asdict(cfg).items() if k != "ranking"}, **asdict(cfg.ranking)}
    for name, cfg in MODES.items()
})

,임베딩만,조건적용,조건+재랭킹+중복제어
top_k,5,5,5
candidate_k,100,100,100
preference_widen_k,400,400,400
apply_filters,False,True,True
similarity_weight,1.0,1.0,0.7
preference_weight,0.0,0.0,0.3
group_key,대표식품명,대표식품명,대표식품명
group_cap,0,0,2
group_penalty,0.0,0.0,0.0
collapse_duplicates,False,False,True


## 6. 단일 질의 상세 흐름

조건 추출 → 후보 100개 검색 → 필수 필터 → 재랭킹 → 중복·상한 → Top-5.
필수 조건 통과 항목이 부족하면 전체까지, 선호를 모두 만족하는 항목이 부족하면 400까지 검색 범위를 두 배씩 넓힌다 (필수 조건은 완화하지 않음). `확장사유`에 어느 쪽인지 남는다.

In [8]:
SHOW_COLS = ["순위", "메뉴명", "업체명", "대표식품명", "주요라벨", "유사도", "선호점수", "최종점수", "추천근거"]


def show(result):
    print(f"질의: {result['질의']!r}  상태={result['상태']}  사유={result['사유']}")
    print(f"검색범위={result['검색범위']} (확장사유={result['확장사유']})  후보={result['후보수']}  "
          f"필터통과={result['필터통과']}  필터제외={result['필터제외']} {result['필터제외사유']}")
    print(f"반환 {result['반환수']}/{result['요청수']}  실행시간 {result['실행시간'].get('전체', 0):.3f}초  "
          f"임베딩={result['임베딩']['name']}")
    return pd.DataFrame(result["추천"], columns=SHOW_COLS)


detail = rec_b.recommend("국물 없는 매운 음식")
print("조건:", parse_query(detail["질의"]).summary())
show(detail)

조건: 필수 국물∈국물없음(국물 없는) | 선호 매운맛∈보통/강함(매운)
질의: '국물 없는 매운 음식'  상태=ok  사유=None
검색범위=[100] (확장사유=None)  후보=100  필터통과=27  필터제외=73 {'국물=국물요리': 67, '국물=국물약간': 6}
반환 5/5  실행시간 0.018초  임베딩=multilingual-e5-base_d1287505_textB_v1_f87e895f


,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
0,1,매운 양념 치킨,호식이두마리치킨,닭튀김,"매운맛 강함, 국물없음, 제공온도 뜨거움, 조리법 튀김, 기름짐 높음",0.8567,1.0,0.8997,유사도 0.8567 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=강함 일치(매운)
1,2,쟁반국수,-,쟁반국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8543,1.0,0.8980,유사도 0.8543 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
2,3,매운불고기 피자,피자와치킨의러브레터,피자,"매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음",0.8539,1.0,0.8977,유사도 0.8539 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
3,4,직화매운갈비 피자,선명희피자,피자,"매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음",0.8529,1.0,0.8970,유사도 0.8529 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
4,5,막국수,-,막국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8497,1.0,0.8948,유사도 0.8497 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)


In [9]:
# 중복·상한으로 빠진 후보
pd.DataFrame(detail["제외"])

,메뉴명,업체명,최종점수,제외사유
0,매운양념 치킨 반마리,비비큐,0.8987,중복: 매운 양념 치킨
1,매운양념 치킨,비비큐,0.8976,중복: 매운 양념 치킨
2,직화매운갈비피자,선명희피자,0.8952,중복: 직화매운갈비 피자


### 후보 부족 시 동작

필수 조건 통과 후보가 부족하면 검색 범위를 100 → 200 → 400 → ... 전체까지 넓힌다. 필수 조건은 완화하지 않는다.
전체를 훑어도 부족하면 상태를 `shortage`로 표시하고 실제 반환 개수와 사유를 남긴다. 첫 예는 범위 확장만으로 채워지는 경우, 둘째 예는 요청 개수를 60으로 올려 전체를 훑어도 부족한 경우다 (차가움 라벨은 전체 72건뿐이다).

In [10]:
widened = rec_b.recommend("꼭 차가운 찜 요리")
print("조건:", parse_query(widened["질의"]).summary())
show(widened)

조건: 필수 제공온도∈차가움(차가운) | 선호 조리법∈찜(찜)
질의: '꼭 차가운 찜 요리'  상태=ok  사유=None
검색범위=[100, 200, 400, 800] (확장사유=필수 조건 통과 후보 부족)  후보=800  필터통과=15  필터제외=785 {'제공온도=뜨거움': 754, '제공온도=따뜻함': 26, '제공온도=미확인': 5}
반환 5/5  실행시간 0.019초  임베딩=multilingual-e5-base_d1287505_textB_v1_f87e895f


,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
0,1,냉면 회냉면 홍어,-,냉면,"매운맛 보통, 국물약간, 제공온도 차가움, 조리법 혼합, 기름짐 낮음, 든든함 보통",0.8359,0.0,0.5851,유사도 0.8359 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=혼합 불일치(찜)
1,2,가지냉국,-,가지냉국,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 가벼움",0.8352,0.0,0.5846,유사도 0.8352 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(찜)
2,3,콩국수,-,콩국수,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 보통, 든든함 보통",0.8351,0.0,0.5845,유사도 0.8351 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(찜)
3,4,연어롤,-,연어롤,"매운맛 없음, 국물없음, 제공온도 차가움, 조리법 비조리, 기름짐 보통",0.8351,0.0,0.5845,유사도 0.8351 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(찜)
4,5,물냉면,-,물냉면,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8348,0.0,0.5844,유사도 0.8348 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(찜)


In [11]:
short = rec_b.recommend("무조건 차가운 튀김", PipelineConfig(top_k=60))
print("조건:", parse_query(short["질의"]).summary())
show(short).tail(5)

조건: 필수 제공온도∈차가움(차가운) | 선호 조리법∈튀김(튀김)
질의: '무조건 차가운 튀김'  상태=shortage  사유=전체 5219건 중 필수 조건 통과 72건, 중복·상한 제외 후 34건만 남음
검색범위=[100, 200, 400, 800, 1600, 3200, 5219] (확장사유=필수 조건 통과 후보 부족)  후보=5219  필터통과=72  필터제외=5147 {'제공온도=뜨거움': 4237, '제공온도=따뜻함': 705, '제공온도=미확인': 103, '제공온도=상온': 102}
반환 34/60  실행시간 0.036초  임베딩=multilingual-e5-base_d1287505_textB_v1_f87e895f


,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
29,30,간편조리세트 매콤제육비빔면,신세계푸드 피코크,비빔면,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 보통",0.8038,0.0,0.5627,유사도 0.8038 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(튀김)
30,31,우럭회덮밥 양념장,-,우럭회덮밥,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 보통",0.8015,0.0,0.5611,유사도 0.8015 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(튀김)
31,32,밀면 물밀면,-,밀면,"매운맛 약함, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8008,0.0,0.5606,유사도 0.8008 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(튀김)
32,33,묵말이 도토리묵,-,묵말이,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 가벼움",0.7980,0.0,0.5586,유사도 0.7980 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(튀김)
33,34,묵국 메밀묵,-,묵국,"매운맛 약함, 국물요리, 제공온도 차가움, 조리법 혼합, 기름짐 낮음, 든든함 가벼움",0.7954,0.0,0.5568,유사도 0.7954 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=혼합 불일치(튀김)


## 7. 비교 실험 (텍스트 B)

4단계의 12개 질의에 부정·복합·모순·미확정·빈 입력 사례를 더해 세 모드를 비교한다.
지표 정의 (모두 저장된 추정 라벨 기준, 반환된 항목만 셈)
- 조건위반수: 필수 조건 허용값에 없는 라벨('미확인' 포함)을 가진 항목 수
- 미확인포함수: 질의가 언급한 속성 중 하나라도 '미확인'인 항목 수
- 선호불일치수: 선호 조건에 맞지 않는 (미확인 제외) 항목·조건 쌍 수
- 중복메뉴수: 앞 항목과 같은 메뉴 변형으로 판정된 항목 수
- 대표식품명반복수: 반환수 − 서로 다른 대표식품명 수, 메뉴군반복수: 반환수 − 서로 다른 메뉴군 수 (상한이 실제로 제어하는 단위)

In [12]:
def run_all(rec, queries, modes, tag):
    rows, metrics, results = [], [], {}
    for q in queries:
        for mode, cfg in modes.items():
            r = rec.recommend(q, cfg)
            results[(mode, q)] = r
            rows += result_rows(r, 텍스트구성=tag, 모드=mode, 유형=QUERY_KIND[q])
            metrics.append({"텍스트구성": tag, "모드": mode, "유형": QUERY_KIND[q], **result_metrics(r)})
    return pd.DataFrame(rows), pd.DataFrame(metrics), results


t0 = time.time()
rows_b, metrics_b, results_b = run_all(rec_b, ALL_QUERIES, MODES, "B")
print(f"{len(ALL_QUERIES)}개 질의 × {len(MODES)}개 모드 = {len(metrics_b)}회 실행, 추천 행 {len(rows_b)}개, {time.time() - t0:.1f}초")

22개 질의 × 3개 모드 = 66회 실행, 추천 행 285개, 0.3초


In [13]:
# 질의별 메뉴명·업체·주요 라벨·점수·근거 (전체 파이프라인)
rows_b[rows_b["모드"] == "조건+재랭킹+중복제어"][["유형", "질의", *SHOW_COLS]]

,유형,질의,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
10,기존,비 오는 날 얼큰한 국물 먹고 싶어,1,해장국 뼈다귀,-,해장국,"매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함",0.8271,1.0,0.8789,유사도 0.8271 / 선호 매운맛=보통 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
11,기존,비 오는 날 얼큰한 국물 먹고 싶어,2,국물맵떡,교촌치킨,떡볶이,"매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8254,1.0,0.8778,유사도 0.8254 / 선호 매운맛=강함 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
12,기존,비 오는 날 얼큰한 국물 먹고 싶어,3,꽃게 매운탕,-,꽃게 매운탕,"매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8248,1.0,0.8774,유사도 0.8248 / 선호 매운맛=강함 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
13,기존,비 오는 날 얼큰한 국물 먹고 싶어,4,알탕,-,알탕,"매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함 보통",0.8244,1.0,0.8771,유사도 0.8244 / 선호 매운맛=강함 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
14,기존,비 오는 날 얼큰한 국물 먹고 싶어,5,복 매운탕,-,복 매운탕,"매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.8243,1.0,0.8770,유사도 0.8243 / 선호 매운맛=강함 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
25,기존,맵지 않고 따뜻한 음식,1,꽃맛살쉬림프,봉수아피자,피자,"매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통",0.8450,1.0,0.8915,유사도 0.8450 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=뜨거움 일치(따뜻한)
26,기존,맵지 않고 따뜻한 음식,2,불고기와퍼 버거,버거킹,버거,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 구이, 기름짐 보통, 든든함",0.8435,1.0,0.8905,유사도 0.8435 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)
27,기존,맵지 않고 따뜻한 음식,3,업그레이비타워,KFC,버거,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 혼합, 기름짐 높음, 든든함",0.8431,1.0,0.8902,유사도 0.8431 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)
28,기존,맵지 않고 따뜻한 음식,4,웰빙다이어트야채 피자,왕손피자,피자,"매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통",0.8422,1.0,0.8896,유사도 0.8422 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=뜨거움 일치(따뜻한)
29,기존,맵지 않고 따뜻한 음식,5,수플레 오믈렛 라이스,할리스,오므라이스,"매운맛 없음, 국물약간, 제공온도 따뜻함, 조리법 혼합, 기름짐 보통, 든든함 보통",0.8422,1.0,0.8895,유사도 0.8422 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)


In [14]:
# 세 모드 나란히 보기: 메뉴명 [업체] 최종점수
def compact(rows, modes):
    rows = rows.copy()
    rows["항목"] = rows["메뉴명"] + " [" + rows["업체명"] + "] " + rows["최종점수"].map("{:.3f}".format)
    return rows.pivot(index=["질의", "순위"], columns="모드", values="항목")[list(modes)].fillna("")


compact(rows_b, MODES)

모드                                                        임베딩만                                   조건적용                        조건+재랭킹+중복제어
질의                   순위                                                                                                                 
국물 없는 매운 음식          1                        해물 매운탕 [-] 0.866                매콤 로제 구운주먹밥 [이디야] 0.861          매운 양념 치킨 [호식이두마리치킨] 0.900
                     2                        광어 매운탕 [-] 0.865              매운 양념 치킨 [호식이두마리치킨] 0.857                     쟁반국수 [-] 0.898
                     3                        잉어 매운탕 [-] 0.865                매운양념 치킨 반마리 [비비큐] 0.855        매운불고기 피자 [피자와치킨의러브레터] 0.898
                     4                        메기 매운탕 [-] 0.864                         쟁반국수 [-] 0.854            직화매운갈비 피자 [선명희피자] 0.897
                     5                        버섯 매운탕 [-] 0.863               햄&치즈라우겐샌드위치 [뚜레쥬르] 0.854                      막국수 [-] 0.895
너무 맵지 않은 국물 요리       1                       국물맵떡 [교촌치킨] 0.851                       해물 된장국 [-] 0.846                   해물 된장국 [-] 0.892
                     2                        해물 된장국 [-] 0.846                        굴 미역국 [-] 0.846                    굴 미역국 [-] 0.892
                     3                         굴 미역국 [-] 0.846                          무국물 [-] 0.846                      무국물 [-] 0.892
                     4                           무국물 [-] 0.846                        무 된장국 [-] 0.845                    무 된장국 [-] 0.891
                     5                         무 된장국 [-] 0.845                          선짓국 [-] 0.845                      선짓국 [-] 0.891
느끼하지 않은 담백한 음식       1                    꽃맛살쉬림프 [봉수아피자] 0.844                   꽃맛살쉬림프 [봉수아피자] 0.844      식물성대체육옴미니트샐러드랩 [투썸플레이스] 0.889
                     2            전여친 생각 토스트 [크로플덕오리아가씨] 0.843           전여친 생각 토스트 [크로플덕오리아가씨] 0.843                    무 된장국 [-] 0.887
                     3             바삭담백한 후라이드 치킨 [치킨플러스] 0.843               더블 한우불고기 버거 [롯데리아] 0.842                    홍합 무국 [-] 0.887
                     4                더블 한우불고기 버거 [롯데리아] 0.842                          화양적 [-] 0.842                      백합죽 [-] 0.887
                     5              불닭쉬림프 피자 씬도우 [피자파는집] 0.842                       붕어 매운탕 [-] 0.842                      무국물 [-] 0.886
단짠단짠한 음식             1                   단짠반반 피자 [서오릉피자] 0.861                  단짠반반 피자 [서오릉피자] 0.861              단짠반반 피자 [서오릉피자] 0.603
                     2                      단짠반반 [서오릉피자] 0.859                     단짠반반 [서오릉피자] 0.859           단짠콘후라이 피자 [피자는치즈빨] 0.596
                     3                단짠콘후라이 피자 [피자는치즈빨] 0.852               단짠콘후라이 피자 [피자는치즈빨] 0.852                단짠갈릭 치킨 [비비큐] 0.590
                     4              불닭바베큐 피자 씬도우 [피자파는집] 0.844             불닭바베큐 피자 씬도우 [피자파는집] 0.844              징거더블다운통다리 [KFC] 0.587
                     5                    맵단불고기 피자 [고피자] 0.843                   맵단불고기 피자 [고피자] 0.843                       분짜 [-] 0.586
든든한 밥 한 끼            1                           잡탕밥 [-] 0.832                          잡탕밥 [-] 0.832                      잡탕밥 [-] 0.882
                     2                           잡곡밥 [-] 0.827                          잡곡밥 [-] 0.827                    육회비빔밥 [-] 0.876
                     3                            김밥 [-] 0.827                           김밥 [-] 0.827                      짜장밥 [-] 0.876
                     4                          삼각김밥 [-] 0.826                         삼각김밥 [-] 0.826                   덮밥 닭고기 [-] 0.876
                     5                   달걀듬뿍 볶음밥 [교촌치킨] 0.825                  달걀듬뿍 볶음밥 [교촌치킨] 0.825     간편조리세트 부채살 찹스테이크 [CJ 쿡킷] 0.875
따뜻한 국이나 찌개           1                          두부찌개 [-] 0.865                         두부찌개 [-] 0.865                     두부찌개 [-] 0.905
                     2                      감자 소고기찌개 [-] 0.863                     감자 소고기찌개 [-] 0.863                 감자 소고기찌개 [-] 0.904
                     3     

In [15]:
METRIC_COLS = ["반환수", "조건위반수", "미확인포함수", "선호불일치수", "중복메뉴수", "대표식품명반복수", "메뉴군반복수", "검색범위", "실행시간초"]
metrics_b.pivot(index=["유형", "질의"], columns="모드", values=METRIC_COLS[:7]).reindex(columns=list(MODES), level=1)

반환수                  조건위반수                  미확인포함수                  선호불일치수  ...             중복메뉴수                  대표식품명반복수                  메뉴군반복수                 
모드                        임베딩만 조건적용 조건+재랭킹+중복제어  임베딩만 조건적용 조건+재랭킹+중복제어   임베딩만 조건적용 조건+재랭킹+중복제어   임베딩만  ... 조건+재랭킹+중복제어  임베딩만 조건적용 조건+재랭킹+중복제어     임베딩만 조건적용 조건+재랭킹+중복제어   임베딩만 조건적용 조건+재랭킹+중복제어
유형   질의                                                                                                ...                                                                                     
기존   국물 없는 매운 음식             5    5           5     5    0           0      0    0           0      0  ...           0     0    1           0        0    1           1      4    1           1
     느끼하지 않은 담백한 음식          5    5           5     2    0           0      1    1           0      6  ...           0     0    0           0        1    0           0      1    0           0
     단짠단짠한 음식                5    5           5     0    0           0      0    0           0      0  ...           0     1    1           0        4    4           1      4    4           1
     든든한 밥 한 끼               5    5           5     0    0           0      1    1           0      3  ...           0     0    0           0        0    0           0      0    0           0
     따뜻한 국이나 찌개              5    5           5     0    0           0      0    0           0      0  ...           0     1    1           0        1    1           0      2    2           1
     맵지 않고 따뜻한 음식            5    5           5     1    0           0      4    3           0      0  ...           0     0    0           0        2    2           2      2    2           2
     바삭하고 기름진 음식             5    5           5     0    0           0      0    0           0      7  ...           0     2    2           0        4    4           1      4    4           1
     비 오는 날 얼큰한 국물 먹고 싶어     5    5           5     0    0           0      0    0           0      5  ...           0     1    1           0        0    0           0      0    0           1
     빠르게 먹을 수 있는 간식          5    5           5     0    0           0      2    2           0      2  ...           0     0    0           0        2    2           0      2    2           0
     상큼하고 시원한 음식             5    5           5     0    0           0      0    0           0      5  ...           3     0    0           0        1    1           2      1    1           2
     차갑고 가볍게 먹을 메뉴           5    5           5     0    0           0      3    3           3      7  ...           3     0    0           0        1    1           1      1    1           1
     포만감 있는 저녁밥              5    5           5     0    0           0      1    1           0      4  ...           0     0    0           0        0    0           2      1    1           3
모순   국물 없는 국물 요리             0    0           0     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      0    0           0
     맵지 않은 매운 음식             0    0           0     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      0    0           0
미확정  매운 것도 괜찮아               5    5           5     0    0           0      0    0           0      0  ...           0     2    2           0        3    3           1      3    3           2
     안 매운 건 싫어               5    5           5     0    0           0      0    0           0      0  ...           0     1    1           0        1    1           0      2    2           1
복합   너무 맵지 않은 국물 요리          5    5           5     1    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      1    1           1
     차가운 국물 요리               5    5           5     0    0           0      0    0           0      5  ...           0     0    0           0        0    0           0      0    0        

In [16]:
AGG = {"반환수": "sum", "조건위반수": "sum", "미확인포함수": "sum", "선호불일치수": "sum",
       "중복메뉴수": "sum", "대표식품명반복수": "sum", "메뉴군반복수": "sum", "실행시간초": "mean"}
summary_b = metrics_b.groupby("모드").agg(AGG).loc[list(MODES)]
summary_b

,반환수,조건위반수,미확인포함수,선호불일치수,중복메뉴수,대표식품명반복수,메뉴군반복수,실행시간초
모드,,,,,,,,
임베딩만,95,21,12,48,10,30,38,0.010923
조건적용,95,0,11,45,9,29,33,0.000705
조건+재랭킹+중복제어,95,0,3,6,0,20,26,0.001032


In [17]:
# 정상 반환이 아닌 질의: 모순, 빈 입력, 후보 부족
mask = (metrics_b["모드"] == "조건+재랭킹+중복제어") & (metrics_b["상태"] != "ok")
for _, m in metrics_b[mask].iterrows():
    r = results_b[("조건+재랭킹+중복제어", m["질의"])]
    print(f"[{m['상태']}] {m['질의']!r}: 반환 {m['반환수']}/{m['요청수']} — {r['사유']}")

[contradiction] '맵지 않은 매운 음식': 반환 0/5 — 필수 조건이 서로 모순됨: 매운맛(맵지 않은 vs 매운)
[contradiction] '국물 없는 국물 요리': 반환 0/5 — 필수 조건이 서로 모순됨: 국물(국물 없는 vs 국물)
[empty_query] '': 반환 0/5 — 입력이 비어 있음


## 8. 텍스트 A 기준선

A(음식명+대표식품명+분류)와 B(A+속성 라벨)를 같은 파이프라인으로 비교한다. 어느 쪽이 더 정확하다고 단정하지 않으며 관찰 수치만 남긴다.

In [18]:
rows_a, metrics_a, results_a = run_all(rec_a, ALL_QUERIES, {"임베딩만": EMBEDDING_ONLY, "조건+재랭킹+중복제어": FULL}, "A")
summary_ab = pd.concat([metrics_a, metrics_b]).groupby(["텍스트구성", "모드"]).agg(AGG)
summary_ab

반환수  조건위반수  미확인포함수  선호불일치수  중복메뉴수  대표식품명반복수  메뉴군반복수     실행시간초
텍스트구성 모드                                                                        
A     임베딩만          95     21      16      42      6        30      41  0.011259
      조건+재랭킹+중복제어   95      0       0       4      0        17      23  0.001068
B     임베딩만          95     21      12      48     10        30      38  0.010923
      조건+재랭킹+중복제어   95      0       3       6      0        20      26  0.001032
      조건적용          95      0      11      45      9        29      33  0.000705

In [19]:
# A/B 전체 파이프라인 1순위가 같은 질의 수
valid = [q for q in ALL_QUERIES if results_b[("조건+재랭킹+중복제어", q)]["추천"]]
same = [q for q in valid
        if results_a[("조건+재랭킹+중복제어", q)]["추천"] and
        results_a[("조건+재랭킹+중복제어", q)]["추천"][0]["라벨링단위ID"] == results_b[("조건+재랭킹+중복제어", q)]["추천"][0]["라벨링단위ID"]]
print(f"A/B 1순위 동일: {len(same)}/{len(valid)}건")
for q in valid:
    a = results_a[("조건+재랭킹+중복제어", q)]["추천"]
    b = results_b[("조건+재랭킹+중복제어", q)]["추천"]
    print(f"  {q!r}: A={a[0]['메뉴명'] if a else '-'} / B={b[0]['메뉴명'] if b else '-'}")

A/B 1순위 동일: 3/19건
  '비 오는 날 얼큰한 국물 먹고 싶어': A=김치 된장국 / B=해장국 뼈다귀
  '맵지 않고 따뜻한 음식': A=볶음 우동 / B=꽃맛살쉬림프
  '차갑고 가볍게 먹을 메뉴': A=쿠키 & 크림 샌디 / B=크랜베리 치킨 치즈 샌드위치
  '바삭하고 기름진 음식': A=바삭몬테크리스토 / B=아빠의제주깜슐랭 치킨
  '든든한 밥 한 끼': A=간편조리세트 닭한마리와 칼국수 / B=잡탕밥
  '국물 없는 매운 음식': A=매운불고기 피자 / B=매운 양념 치킨
  '상큼하고 시원한 음식': A=쫄면 / B=씨앗곡물튜나샌드
  '포만감 있는 저녁밥': A=하이라이스 / B=소고기 덮밥
  '빠르게 먹을 수 있는 간식': A=다이어트싱글 버거 / B=다이어트싱글 버거
  '따뜻한 국이나 찌개': A=국수전골 / B=두부찌개
  '느끼하지 않은 담백한 음식': A=무 된장국 / B=식물성대체육옴미니트샐러드랩
  '단짠단짠한 음식': A=단짠반반 / B=단짠반반 피자
  '매운 거 싫어': A=미음 / B=새우에빠진닭 피자
  '튀김 말고 구운 치킨': A=불금 치킨 / B=불금 치킨
  '피자 먹고 싶은데 느끼하지 않은 걸로': A=반미터 생각나는 피자 / B=땡초참치마요 피자
  '차가운 국물 요리': A=미역냉국 / B=국수 김치말이국수
  '너무 맵지 않은 국물 요리': A=무 된장국 / B=해물 된장국
  '안 매운 건 싫어': A=미음 / B=복 매운탕
  '매운 것도 괜찮아': A=매운 간장 치킨 / B=매운 간장 치킨


## 9. 중복·다양성 설정 실험

메뉴군 상한(0=없음, 1, 2, 3)과 그룹 감점(0, 0.02)을 조합해 반복 수와 부족 질의 수를 본다.
모순·빈 입력 질의는 제외한다. 확장질의수는 선호 일치 부족으로 후보를 100개 넘게 넓힌 질의 수다.

In [20]:
def experiment(rec, queries, configs):
    out = []
    for name, cfg in configs.items():
        df = pd.DataFrame([result_metrics(rec.recommend(q, cfg)) for q in queries])
        out.append({
            "설정": name, "반환수합": df["반환수"].sum(), "조건위반합": df["조건위반수"].sum(),
            "미확인합": df["미확인포함수"].sum(), "선호불일치합": df["선호불일치수"].sum(),
            "중복메뉴합": df["중복메뉴수"].sum(), "대표식품명반복합": df["대표식품명반복수"].sum(),
            "메뉴군반복합": df["메뉴군반복수"].sum(), "확장질의수": int((df["검색범위"] > 100).sum()),
            "부족질의수": int((df["상태"] == "shortage").sum()), "평균실행초": round(df["실행시간초"].mean(), 4),
        })
    return pd.DataFrame(out).set_index("설정")


VALID_QUERIES = [q for q in ALL_QUERIES if q and not parse_query(q).contradictions]

diversity_configs = {"중복제거·상한 없음": PipelineConfig(ranking=RankingConfig(collapse_duplicates=False, group_cap=0))}
for cap in (0, 1, 2, 3):
    for penalty in (0.0, 0.02):
        diversity_configs[f"상한{cap} 감점{penalty}"] = PipelineConfig(
            ranking=RankingConfig(group_cap=cap, group_penalty=penalty))
exp_diversity = experiment(rec_b, VALID_QUERIES, diversity_configs)
exp_diversity

,반환수합,조건위반합,미확인합,선호불일치합,중복메뉴합,대표식품명반복합,메뉴군반복합,확장질의수,부족질의수,평균실행초
설정,,,,,,,,,,
중복제거·상한 없음,95,0,2,6,9,32,37,4,0,0.0012
상한0 감점0.0,95,0,2,6,0,24,31,4,0,0.0012
상한0 감점0.02,95,0,2,6,0,16,18,4,0,0.0016
상한1 감점0.0,95,0,2,10,0,10,12,6,0,0.0013
상한1 감점0.02,95,0,2,10,0,10,12,6,0,0.0023
상한2 감점0.0,95,0,3,6,0,20,26,4,0,0.0012
상한2 감점0.02,95,0,3,6,0,15,17,4,0,0.0016
상한3 감점0.0,95,0,2,6,0,23,30,4,0,0.0012
상한3 감점0.02,95,0,2,6,0,16,18,4,0,0.0016


In [21]:
# 사용자가 '피자'를 직접 요청하면 상한을 면제한다
for name in ("상한1 감점0.0", "상한2 감점0.0"):
    r = rec_b.recommend("피자 먹고 싶은데 느끼하지 않은 걸로", diversity_configs[name])
    print(f"{name}: {[it['메뉴명'] for it in r['추천']]}")
r = rec_b.recommend("단짠단짠한 음식", FULL)
print("피자 언급 없는 질의:", [(it["메뉴명"], it["대표식품명"]) for it in r["추천"]])
print("  제외:", [(e["메뉴명"], e["제외사유"]) for e in r["제외"]])

상한1 감점0.0: ['땡초참치마요 피자', '꽃맛살쉬림프 피자', '반미터 생각나는 피자', '불고기 피자', '피자 불고기피자']
상한2 감점0.0: ['땡초참치마요 피자', '꽃맛살쉬림프 피자', '반미터 생각나는 피자', '불고기 피자', '피자 불고기피자']
피자 언급 없는 질의: [('단짠반반 피자', '피자'), ('단짠콘후라이 피자', '피자'), ('단짠갈릭 치킨', '닭튀김'), ('징거더블다운통다리', '버거'), ('분짜', '분짜')]
  제외: [('단짠반반', '중복: 단짠반반 피자'), ('불닭바베큐 피자 씬도우', '대표식품명 상한(2): 피자'), ('맵단불고기 피자', '대표식품명 상한(2): 피자'), ('탄탄불고기 피자 씬도우', '대표식품명 상한(2): 피자'), ('리얼단호박 피자씬도우', '대표식품명 상한(2): 피자'), ('불닭쉬림프 피자 씬도우', '대표식품명 상한(2): 피자'), ('단고통밀도우피자', '대표식품명 상한(2): 피자'), ('꽃맛살쉬림프', '대표식품명 상한(2): 피자'), ('불닭쉬림프 피자씬도우', '대표식품명 상한(2): 피자'), ('웃음꽃 피자', '대표식품명 상한(2): 피자'), ('할라직화불고기피자씬', '대표식품명 상한(2): 피자')]


적용 기준
- 중복 제거만으로 같은 메뉴 변형(브랜드·사이즈·도우 차이)의 반복은 사라진다 (중복메뉴합 0). 이 부분은 설정과 무관하게 항상 켠다.
- 상한은 메뉴군 단위로 걸리므로 메뉴군반복합을 기준으로 본다. 상한 1이 가장 적지만 선호 불일치가 늘 수 있고, 감점 0.02는 반복을 더 줄이면서 선호 불일치를 유지한다.
- 반복을 줄이는 것이 곧 더 좋은 추천이라는 근거(정답 데이터)가 없으므로, 기본값은 설명이 가장 단순한 **중복 제거 + 상한 2 + 감점 0**으로 두고 감점 0.02와 상한 1은 6단계에서 정답 기준으로 판단할 후보로 기록한다.
- 사용자가 언급한 메뉴(대표식품명·메뉴명에 포함)는 상한을 적용하지 않는다. 위 셀에서 "피자" 질의는 상한 1에서도 피자 5개를 돌려준다.

## 10. 선호 가중치 민감도

`preference_weight`를 바꾸며 선호 불일치와 반환 항목의 평균 유사도를 본다.
후보 100개 안의 유사도 차이는 0.05 이하로 작아서, 작은 가중치만 줘도 선호 일치가 유사도 차이를 압도한다. 즉 가중치는 "선호를 반영할지 말지"에 가까운 스위치로 동작한다.

In [22]:
def mean_similarity(rec, queries, cfg):
    sims = [it["유사도"] for q in queries for it in rec.recommend(q, cfg)["추천"]]
    return round(float(np.mean(sims)), 4)


weight_configs = {
    f"선호가중치 {w}": PipelineConfig(ranking=RankingConfig(similarity_weight=1 - w, preference_weight=w))
    for w in (0.0, 0.05, 0.1, 0.3, 0.5)
}
exp_weights = experiment(rec_b, VALID_QUERIES, weight_configs)
exp_weights["평균유사도"] = [mean_similarity(rec_b, VALID_QUERIES, c) for c in weight_configs.values()]
exp_weights

,반환수합,조건위반합,미확인합,선호불일치합,중복메뉴합,대표식품명반복합,메뉴군반복합,확장질의수,부족질의수,평균실행초,평균유사도
설정,,,,,,,,,,,
선호가중치 0.0,95,0,10,46,0,18,23,0,0,0.0008,0.8411
선호가중치 0.05,95,0,3,6,0,20,26,4,0,0.0012,0.8381
선호가중치 0.1,95,0,3,6,0,20,26,4,0,0.0012,0.8381
선호가중치 0.3,95,0,3,6,0,20,26,4,0,0.0012,0.8381
선호가중치 0.5,95,0,3,6,0,20,26,4,0,0.0012,0.8381


## 11. 후보 수 민감도

선호 재랭킹은 검색된 후보 안에서만 동작하므로 후보 수가 결과에 영향을 준다.
여기서는 자동 확장을 끄고(`preference_widen_k=candidate_k`) 후보 수를 고정한 뒤, 마지막 행에서 기본 설정(100에서 시작해 선호 일치 부족 시 400까지 확장)과 비교한다.

In [23]:
k_configs = {f"후보 {k} 고정": PipelineConfig(candidate_k=k, preference_widen_k=k) for k in (50, 100, 200, 400)}
k_configs["기본 (100→400 자동 확장)"] = FULL
exp_k = experiment(rec_b, VALID_QUERIES, k_configs)
exp_k["평균유사도"] = [mean_similarity(rec_b, VALID_QUERIES, c) for c in k_configs.values()]
exp_k

,반환수합,조건위반합,미확인합,선호불일치합,중복메뉴합,대표식품명반복합,메뉴군반복합,확장질의수,부족질의수,평균실행초,평균유사도
설정,,,,,,,,,,,
후보 50 고정,95,0,5,24,0,20,26,0,0,0.0008,0.8401
후보 100 고정,95,0,5,15,0,21,27,0,0,0.0008,0.8394
후보 200 고정,95,0,4,14,0,20,26,19,0,0.0010,0.8392
후보 400 고정,95,0,3,6,0,20,26,19,0,0.0015,0.8381
기본 (100→400 자동 확장),95,0,3,6,0,20,26,4,0,0.0012,0.8381


In [24]:
# 후보 100개 안에 선호 일치 항목이 없던 질의: 고정 100 vs 기본(자동 확장)
for name in ("후보 100 고정", "기본 (100→400 자동 확장)"):
    r = rec_b.recommend("차갑고 가볍게 먹을 메뉴", k_configs[name])
    print(f"{name}: 검색범위={r['검색범위']} 확장사유={r['확장사유']}")
    print("   ", [(it["메뉴명"], it["선호점수"]) for it in r["추천"]])

후보 100 고정: 검색범위=[100] 확장사유=None
    [('간편조리세트 차돌박이숙주볶음', 0.0), ('간편조리세트 대파고추장불고기', 0.0), ('할라불고기 피자', 0.0), ('간편조리세트 소고기야채말이', 0.0), ('간편조리세트 매콤 콩나물불고기', 0.0)]
기본 (100→400 자동 확장): 검색범위=[100, 200, 400] 확장사유=선호 조건 일치 후보 부족
    [('크랜베리 치킨 치즈 샌드위치', 0.5), ('치킨 샐러드 통밀 샌드위치', 0.5), ('소고기 감자죽', 0.5), ('간편조리세트 매콤제육비빔면', 0.5), ('간편조리세트 차돌박이숙주볶음', 0.0)]


## 12. 개선 사례: 임베딩만 vs 전체 파이프라인

실제 결과를 나란히 놓고 본다. 아래 표는 4단계에서 확인한 한계(부정 조건 무시, 미확인 라벨 노출, 같은 메뉴 반복)가 어떻게 달라졌는지 보여준다.

In [25]:
def side_by_side(q):
    fmt = lambda r: [f"{it['메뉴명']} [{it['업체명']}] — {it['주요라벨']}" for it in r["추천"]]
    left, right = fmt(results_b[("임베딩만", q)]), fmt(results_b[("조건+재랭킹+중복제어", q)])
    n = max(len(left), len(right))
    return pd.DataFrame({"임베딩만": left + [""] * (n - len(left)),
                         "조건+재랭킹+중복제어": right + [""] * (n - len(right))}, index=range(1, n + 1))


for q in ["맵지 않고 따뜻한 음식", "국물 없는 매운 음식", "차가운 국물 요리", "단짠단짠한 음식", "느끼하지 않은 담백한 음식"]:
    print(f"\n### {q}  |  {parse_query(q).summary()}")
    display(side_by_side(q))


### 맵지 않고 따뜻한 음식  |  필수 매운맛∈없음(맵지 않고) | 선호 제공온도∈뜨거움/따뜻함(따뜻한)


,임베딩만,조건+재랭킹+중복제어
1,"전여친 생각 토스트 [크로플덕오리아가씨] — 국물없음, 제공온도 따뜻함, 조리법 구이, 기름짐 보통","꽃맛살쉬림프 [봉수아피자] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통"
2,"햄&치즈 프레시 샌드위치 [할리스] — 매운맛 없음, 국물없음, 조리법 비조리, 기름짐 보통, 든든함 보통","불고기와퍼 버거 [버거킹] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 구이, 기름짐 보통, 든든함"
3,"꽃맛살쉬림프 [봉수아피자] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통","업그레이비타워 [KFC] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 혼합, 기름짐 높음, 든든함"
4,"게살듬뿍모닝 샌드위치 [바나프레소] — 매운맛 없음, 국물없음, 조리법 비조리, 기름짐 보통, 든든함 보통","웰빙다이어트야채 피자 [왕손피자] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통"
5,"그릴치킨&당근라페샌드위치 [뚜레쥬르] — 매운맛 없음, 국물없음, 조리법 비조리, 기름짐 보통","수플레 오믈렛 라이스 [할리스] — 매운맛 없음, 국물약간, 제공온도 따뜻함, 조리법 혼합, 기름짐 보통, 든든함 보통"



### 국물 없는 매운 음식  |  필수 국물∈국물없음(국물 없는) | 선호 매운맛∈보통/강함(매운)


,임베딩만,조건+재랭킹+중복제어
1,"해물 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통","매운 양념 치킨 [호식이두마리치킨] — 매운맛 강함, 국물없음, 제공온도 뜨거움, 조리법 튀김, 기름짐 높음"
2,"광어 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통","쟁반국수 [-] — 매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통"
3,"잉어 매운탕 [-] — 매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음","매운불고기 피자 [피자와치킨의러브레터] — 매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음"
4,"메기 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통","직화매운갈비 피자 [선명희피자] — 매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음"
5,"버섯 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음","막국수 [-] — 매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통"



### 차가운 국물 요리  |  선호 국물∈국물요리(국물) | 선호 제공온도∈차가움(차가운)


,임베딩만,조건+재랭킹+중복제어
1,"게국지 [-] — 매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통","국수 김치말이국수 [-] — 매운맛 보통, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 보통"
2,"국수 막국수 [-] — 매운맛 약함, 국물약간, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통","콩국수 [-] — 매운맛 없음, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 보통, 든든함 보통"
3,"냉이 된장국 [-] — 매운맛 약함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움","냉국 미역 [-] — 매운맛 없음, 국물요리, 제공온도 차가움, 조리법 혼합, 기름짐 낮음"
4,"선짓국 [-] — 매운맛 약함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음","냉면 열무냉면 [-] — 매운맛 약함, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통"
5,"해물 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통","가지냉국 [-] — 매운맛 없음, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 가벼움"



### 단짠단짠한 음식  |  미처리 '단짠'


,임베딩만,조건+재랭킹+중복제어
1,"단짠반반 피자 [서오릉피자] — 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음","단짠반반 피자 [서오릉피자] — 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음"
2,"단짠반반 [서오릉피자] — 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음","단짠콘후라이 피자 [피자는치즈빨] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음"
3,"단짠콘후라이 피자 [피자는치즈빨] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음","단짠갈릭 치킨 [비비큐] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 튀김, 기름짐 높음"
4,"불닭바베큐 피자 씬도우 [피자파는집] — 매운맛 강함, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통","징거더블다운통다리 [KFC] — 매운맛 보통, 국물없음, 제공온도 따뜻함, 조리법 튀김, 기름짐 높음, 든든함"
5,"맵단불고기 피자 [고피자] — 매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음, 든든함 보통","분짜 [-] — 매운맛 약함, 국물약간, 제공온도 따뜻함, 조리법 혼합, 기름짐 보통, 든든함 보통"



### 느끼하지 않은 담백한 음식  |  필수 기름짐∈낮음/보통(느끼하지 않은) | 선호 기름짐∈낮음(담백한) | 선호 매운맛∈없음/약함(담백한)


,임베딩만,조건+재랭킹+중복제어
1,"꽃맛살쉬림프 [봉수아피자] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 보통","식물성대체육옴미니트샐러드랩 [투썸플레이스] — 매운맛 없음, 국물없음, 조리법 비조리, 기름짐 낮음, 든든함 보통"
2,"전여친 생각 토스트 [크로플덕오리아가씨] — 국물없음, 제공온도 따뜻함, 조리법 구이, 기름짐 보통","무 된장국 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움"
3,"바삭담백한 후라이드 치킨 [치킨플러스] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 튀김, 기름짐 높음, 든든함","홍합 무국 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음"
4,"더블 한우불고기 버거 [롯데리아] — 매운맛 없음, 국물없음, 제공온도 뜨거움, 조리법 구이, 기름짐 보통, 든든함","백합죽 [-] — 매운맛 없음, 국물약간, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움"
5,"불닭쉬림프 피자 씬도우 [피자파는집] — 매운맛 강함, 국물없음, 제공온도 뜨거움, 조리법 오븐, 기름짐 높음","무국물 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음"


In [26]:
changed = [q for q in VALID_QUERIES
           if [it["라벨링단위ID"] for it in results_b[("임베딩만", q)]["추천"]]
           != [it["라벨링단위ID"] for it in results_b[("조건+재랭킹+중복제어", q)]["추천"]]]
print(f"Top-5 구성이 달라진 질의: {len(changed)}/{len(VALID_QUERIES)}건")
unchanged = [q for q in VALID_QUERIES if q not in changed]
print("달라지지 않은 질의:", unchanged)

Top-5 구성이 달라진 질의: 19/19건
달라지지 않은 질의: []


## 13. 결과 저장

`data/processed/recommendation/`에 추천 결과, 지표, 파싱 결과, 실험 표, 실행 설정을 저장한다.
`run_config.json`에 사용한 임베딩 manifest 이름·config_hash·원본 해시와 모드 설정을 남겨 추적할 수 있게 한다.

In [27]:
run_config = {
    "실행시각": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "환경": env,
    "모델": DEFAULT_SPEC.as_dict(),
    "임베딩": {"A": ref_a, "B": ref_b},
    "원본해시": sources,
    "모드설정": {name: asdict(cfg) for name, cfg in MODES.items()},
    "채택설정": asdict(FULL),
    "질의": [{"질의": q, "유형": QUERY_KIND[q]} for q in ALL_QUERIES],
    "지표주의": "조건 준수 지표는 저장된 모델 추정 라벨 기준, 정답 데이터 없음",
    "실험": {
        "다양성": exp_diversity.reset_index().to_dict("records"),
        "가중치": exp_weights.reset_index().to_dict("records"),
        "후보수": exp_k.reset_index().to_dict("records"),
    },
}

pd.concat([rows_b, rows_a], ignore_index=True).to_csv(OUT_DIR / "recommendations.csv", index=False, encoding="utf-8-sig")
pd.concat([metrics_b, metrics_a], ignore_index=True).to_csv(OUT_DIR / "metrics.csv", index=False, encoding="utf-8-sig")
parsed_df.to_csv(OUT_DIR / "parsed_queries.csv", index=False, encoding="utf-8-sig")
with open(OUT_DIR / "parsed_queries.json", "w", encoding="utf-8") as f:
    json.dump({q: parse_query(q).to_dict() for q in ALL_QUERIES}, f, ensure_ascii=False, indent=2)
with open(OUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2, default=str)
with open(OUT_DIR / "results_B_full.json", "w", encoding="utf-8") as f:
    json.dump({f"{mode}|{q}": r for (mode, q), r in results_b.items()}, f, ensure_ascii=False, indent=1, default=str)

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size:>10,} bytes")

metrics.csv                      10,680 bytes
parsed_queries.csv                2,764 bytes
parsed_queries.json              12,220 bytes
recommendations.csv             169,955 bytes
results_B_full.json             370,343 bytes
run_config.json                  13,535 bytes


## 14. 요약과 한계

관찰 (위 셀의 실제 출력 기준, 조건 준수는 저장된 추정 라벨 기준)
- 임베딩만으로는 "맵지 않고", "국물 없는" 같은 부정 조건을 걸러내지 못하고 미확인 라벨 항목이 상위에 노출된다. 필수 조건 필터를 넣으면 반환 항목의 조건위반수는 0이 된다 (7절 요약표).
- 선호 재랭킹은 후보 안에서 선호 일치 항목을 앞으로 보낸다 ("차가운 국물 요리": 뜨거운 국 → 냉국·냉면). 후보 100개 안에 선호 일치 항목이 부족한 질의("차갑고 가볍게 먹을 메뉴")는 400까지 자동 확장해 일치 항목을 찾는다 (11절). 그래도 완전 일치가 없으면 부분 일치 항목이 남는다.
- 선호 가중치는 0.05 이상이면 결과가 같다 (10절). 0.3은 "후보 안에서는 선호를 우선한다"는 정책이지 조정된 값이 아니다.
- 중복 제거는 같은 메뉴 변형 반복을 없앴고 메뉴군 상한 2는 부족 질의 없이 반복을 줄였다. 공공 데이터는 대표식품명 마지막 어절로 묶어 "붕어/명태/꽃게 매운탕"이 한 메뉴군으로 잡힌다. 감점 0.02와 상한 1은 반복을 더 줄이지만 채택 근거가 없어 후보로만 기록했다 (9절). 사용자가 언급한 메뉴는 상한을 면제한다.
- 후보 부족 시 검색 범위 확장과 부족 사유 보고가 실제로 동작한다 (6절). 비교 실험에서 필수 조건 부족으로 넓힌 질의는 없고, 선호 일치 부족으로 400까지 넓힌 질의는 소수다 (9절 확장질의수).
- A/B는 관찰 수치만 남겼다. 이번 질의 집합에서는 A가 미확인·선호 불일치·반복 지표 모두 낮았지만, 정답 데이터가 없으므로 어느 쪽이 우수하다고 단정하지 않는다 (8절).

한계
- 라벨이 전량 모델 추정이라 조건 준수는 라벨 정확도에 종속된다. 든든함은 63%가 미확인이라 선호로만 쓴다. 코드로 고칠 수 없는 데이터 한계다.
- 파서는 표(3절)에 있는 표현만 지원한다. 이중 부정, 허용 표현, "시원한 국물", 스키마에 없는 맛은 조건으로 만들지 않는다.
- "담백한"은 기름짐 낮음 + 매운맛 없음/약함, "얼큰한"은 매운맛 + 제공온도처럼 여러 속성을 뜻하는 표현은 규칙 표에 명시한 범위까지만 해석한다. 라벨 매핑은 여전히 거칠어 의미가 어긋난 항목이 올라올 수 있다.
- 메뉴군은 대표식품명 마지막 어절이라는 단순 규칙이라 "국수 김치말이국수"처럼 어절 구성이 다른 이름은 다른 군으로 잡힌다.
- 선호 가중치는 0.05 이상이면 결과가 같아 사실상 켜고 끄는 스위치이며, 자동 확장 상한 400도 실험 표에서 고른 값일 뿐 정답 기준 근거는 없다.

6단계 평가에서 할 일
- 질의별 정답(적합 메뉴 집합) 데이터를 만들고 Precision@K, nDCG 등 실제 지표를 계산한다.
- A/B, 가중치, 후보 수, 상한 설정을 정답 기준으로 비교해 채택 근거를 만든다.
- 승인 샘플 100개로 필터 오류율(라벨 오류로 인한 잘못된 제외)을 추정한다.
- 파서 미처리 표현의 빈도를 실제 사용자 입력에서 수집해 지원 범위 확장 우선순위를 정한다.